<!-- codex-architecture-notes -->
## Architectural Notes

**Purpose:** Extracts raw Oracle tables into parquet files under the project's raw data directory.

**Notebook Shape:** 15 cells (14 code, 1 markdown).

**Inputs / Data Sources:**
- `df_v_crg_student_course = pd.read_sql(query, connection)`
- `df_v_acs_grade = pd.read_sql(query, connection)`
- `df_v_add_student_degree_status = pd.read_sql(query, connection)`
- `df_v_acd_degree_course = pd.read_sql(query, connection)`
- `df_v_crg_student_passed_credit = pd.read_sql(query, connection)`

**Outputs / Side Effects:**
- `df_v_crg_student_course.to_parquet(RAW_DIR / "v_crg_student_course_raw.parquet",index=False)`
- `df_v_acs_grade.to_parquet(RAW_DIR / "v_acs_grade.parquet",index=False)`
- `df_v_add_student_degree_status.to_parquet(RAW_DIR / "v_add_student_degree_status.parquet",index=False)`
- `df_v_acd_degree_course.to_parquet(RAW_DIR / "v_acd_degree_course.parquet",index=False)`
- `df_v_crg_student_passed_credit.to_parquet(RAW_DIR / "v_crg_student_passed_credit.parquet",index=False)`

**Logic Flow:**
1. Open an Oracle connection.
2. Run SQL queries for each required source view.
3. Load results into pandas DataFrames.
4. Write raw parquet snapshots.

**Maintainability Notes:** This notebook depends on database credentials and live source data; extraction dates, SQL, and schema versions should be tracked for reproducibility.


## setup and connect with data base

In [1]:
import oracledb
import pandas as pd

from src.paths import RAW_DIR, assert_data_root
from src.io_utils import save_parquet
from src.db_connect import get_connection

# Data-root guard (governance contract 12): refuse to run against a freshly
# created empty tree before extracting any raw table.
assert_data_root()

connection = get_connection()

In [2]:
cursor = connection.cursor()
cursor.arraysize = 10_000
pd.set_option('display.max_columns', None)

In [3]:
print("RAW_DIR:", RAW_DIR)


RAW_DIR: D:\AI\Real projects\Academic_Advisor\data\raw


In [4]:
VIEW_NAME = "RAS_USER.V_ADD_ACADEMIC_INFO "

needed_columns = [
    "STUDENT_ID",
    "DIPLOMA_GPA",
    "DIPLOMA_TYPE_ID",
    "DIPLOMA_STATE_ID",
    "DIPLOMA_COUNTRY_SL",
    "DIPLOMA_TYPE_SL"
]
columns_str = ", ".join(needed_columns)
query = f"""
SELECT
    {columns_str}
    ACTIVE
FROM {VIEW_NAME}
WHERE ACTIVE = 'A'
"""

df_v_add_academic_info = pd.read_sql(query, connection)
df_v_add_academic_info.columns = df_v_add_academic_info.columns.str.lower()

save_parquet(df_v_add_academic_info, RAW_DIR / "v_add_academic_info.parquet")

print("Rows loaded:", len(df_v_add_academic_info))
print("Columns:", df_v_add_academic_info.columns.tolist())
print("Saved:", RAW_DIR / "v_add_academic_info.parquet")
display(df_v_add_academic_info.head())

C:\Users\ASUS\AppData\Local\Temp\ipykernel_20092\401297006.py:20: UserWarning: pandas only supports SQLAlchemy connectable (engine/connection) or database string URI or sqlite3 DBAPI2 connection. Other DBAPI2 objects are not tested. Please consider using SQLAlchemy.
  df_v_add_academic_info = pd.read_sql(query, connection)


Rows loaded: 32548
Columns: ['student_id', 'diploma_gpa', 'diploma_type_id', 'diploma_state_id', 'diploma_country_sl', 'active']
Saved: D:\AI\Real projects\Academic_Advisor\data\raw\v_add_academic_info.parquet


,student_id,diploma_gpa,diploma_type_id,diploma_state_id,diploma_country_sl,active
0,1.111,60.00,13.111,13.0,سورية,شهادة ثانوية تجارية
1,2.111,56.82,16.111,15.0,سورية,شهادة ثانوية أدبي
2,3.111,60.45,16.111,4.0,سورية,شهادة ثانوية أدبي
3,4.111,50.83,15.111,4.0,سورية,شهادة ثانوية علمي
4,5.111,67.27,13.111,5.0,سورية,شهادة ثانوية تجارية


In [6]:
VIEW_NAME = "RAS_USER.V_CRG_STUDENT_COURSE"

needed_columns = [
    "STUDENT_COURSE_ID",
    "STUDENT_ID",
    "COURSE_ID",
    "PART_ID",
    "GRADE_ID",
    "FINAL_MARK",
    "POINTS",
    "FINISH_STATUS",
    "COURSE_NAME_SL",
    "REGISTER_STATUS",
    "STUDY_MODE",
    "DEGREE_ID",
    "DEGREE_NAME_SL",
    "FACULTY_ID",
    "COURSE_CREDITS",
    "ACTIVE",
]

query = f"""
SELECT
    STUDENT_COURSE_ID,
    STUDENT_ID,
    COURSE_ID,
    PART_ID,
    GRADE_ID,
    FINAL_MARK,
    POINTS,
    FINISH_STATUS,
    COURSE_NAME_SL,
    REGISTER_STATUS,
    IN_CREDITS,
    IN_GPA,
    IN_AGPA,
    STUDY_MODE,
    DEGREE_ID,
    STUDENT_NAME_SL,
    DEGREE_NAME_SL,
    FACULTY_ID,
    COURSE_CREDITS,
    ACTIVE
FROM {VIEW_NAME}
WHERE ACTIVE = 'A'
  AND STUDY_MODE = 'C'
"""

df_v_crg_student_course = pd.read_sql(query, connection)
df_v_crg_student_course.columns = df_v_crg_student_course.columns.str.lower()

save_parquet(df_v_crg_student_course, RAW_DIR / "v_crg_student_course_raw.parquet")

print("Rows loaded:", len(df_v_crg_student_course))
print("Columns:", df_v_crg_student_course.columns.tolist())
print("Saved:", RAW_DIR / "v_crg_student_course_raw.parquet")
display(df_v_crg_student_course.head())

C:\Users\ASUS\AppData\Local\Temp\ipykernel_20092\707098291.py:49: UserWarning: pandas only supports SQLAlchemy connectable (engine/connection) or database string URI or sqlite3 DBAPI2 connection. Other DBAPI2 objects are not tested. Please consider using SQLAlchemy.
  df_v_crg_student_course = pd.read_sql(query, connection)


Rows loaded: 1017491
Columns: ['student_course_id', 'student_id', 'course_id', 'part_id', 'grade_id', 'final_mark', 'points', 'finish_status', 'course_name_sl', 'register_status', 'in_credits', 'in_gpa', 'in_agpa', 'study_mode', 'degree_id', 'student_name_sl', 'degree_name_sl', 'faculty_id', 'course_credits', 'active']
Saved: D:\AI\Real projects\Academic_Advisor\data\raw\v_crg_student_course_raw.parquet


,student_course_id,student_id,course_id,part_id,grade_id,final_mark,points,finish_status,course_name_sl,register_status,in_credits,in_gpa,in_agpa,study_mode,degree_id,student_name_sl,degree_name_sl,faculty_id,course_credits,active
0,430311.111,2117.111,962.111,20132.0,679.111,70.0,2.50,P,اللغة الانكليزية 2,R,Y,Y,Y,C,13.111,محمد نبيل حلواني,الصيدلة و الكيمياء الصيدلية,4.111,2.0,A
1,430312.111,2117.111,966.111,20132.0,680.111,68.0,2.25,P,كيمياء عضوية صيدلانية 1,R,Y,Y,Y,C,13.111,محمد نبيل حلواني,الصيدلة و الكيمياء الصيدلية,4.111,3.0,A
2,430313.111,2117.111,967.111,20132.0,680.111,69.0,2.25,P,مهارات الحاسوب,R,Y,Y,Y,C,13.111,محمد نبيل حلواني,الصيدلة و الكيمياء الصيدلية,4.111,3.0,A
3,430314.111,2117.111,963.111,20132.0,684.111,48.0,0.00,F,بيولوجيا نباتية,R,Y,Y,Y,C,13.111,محمد نبيل حلواني,الصيدلة و الكيمياء الصيدلية,4.111,3.0,A
4,430315.111,2117.111,964.111,20132.0,679.111,70.0,2.50,P,إحصاء حيوي,R,Y,Y,Y,C,13.111,محمد نبيل حلواني,الصيدلة و الكيمياء الصيدلية,4.111,2.0,A


In [7]:
VIEW_NAME = "RAS_USER.V_ACS_GRADE"

needed_columns = [
    "GRADE_ID",
    "GRADE_VERSION_ID",
    "VERSION_TITLE_SL",
    "VERSION_NUMBER",
    "FROM_PERCENT",
    "TO_PERCENT",
    "POINTS",
    "FINISH_STATUS",
    "GRADE_SHOW",
    "FROM_SEMESTER_ID",
    "TILL_SEMESTER_ID",
    "ACTIVE",
]

query = f"""
SELECT
    GRADE_ID,
    GRADE_VERSION_ID,
    GRADE_NAME_SL,
    VERSION_TITLE_SL,
    VERSION_NUMBER,
    FROM_PERCENT,
    TO_PERCENT,
    POINTS,
    FINISH_STATUS,
    GRADE_SHOW,
    ACTIVE
FROM {VIEW_NAME}
WHERE ACTIVE = 'A'
"""

df_v_acs_grade = pd.read_sql(query, connection)
df_v_acs_grade.columns = df_v_acs_grade.columns.str.lower()

save_parquet(df_v_acs_grade, RAW_DIR / "v_acs_grade.parquet")

print("Rows loaded:", len(df_v_acs_grade))
print("Columns:", df_v_acs_grade.columns.tolist())
print("Saved:", RAW_DIR / "v_acs_grade.parquet")

display(df_v_acs_grade.head())

C:\Users\ASUS\AppData\Local\Temp\ipykernel_20092\138090598.py:35: UserWarning: pandas only supports SQLAlchemy connectable (engine/connection) or database string URI or sqlite3 DBAPI2 connection. Other DBAPI2 objects are not tested. Please consider using SQLAlchemy.
  df_v_acs_grade = pd.read_sql(query, connection)


Rows loaded: 72
Columns: ['grade_id', 'grade_version_id', 'grade_name_sl', 'version_title_sl', 'version_number', 'from_percent', 'to_percent', 'points', 'finish_status', 'grade_show', 'active']
Saved: D:\AI\Real projects\Academic_Advisor\data\raw\v_acs_grade.parquet


,grade_id,grade_version_id,grade_name_sl,version_title_sl,version_number,from_percent,to_percent,points,finish_status,grade_show,active
0,673.111,2.111,شرف,القرار الوزاري الموحد,2,98,100,4.00,P,A+,A
1,999.111,1.111,NaN,تأسيس الجامعة,1,0,0,0.00,NaN,NaN,A
2,974.111,3.111,شرف,القرار رقم 230,230,98,100,4.00,P,A+,A
3,975.111,3.111,شرف,القرار رقم 230,230,95,97,3.75,P,A,A
4,85.111,1.111,ممتاز,تأسيس الجامعة,1,90,100,4.00,P,A,A


In [8]:
VIEW_NAME = "RAS_USER.V_ADD_STUDENT_DEGREE_STATUS"

needed_columns = [
    "STUDENT_ID",
    "PART_ID",
    "DEGREE_ID",
    "STUDY_MODE",
    "GPA_PERCENT",
    "GPA_POINTS",
    "START_AGPA_PERCENT",
    "START_AGPA_POINTS",
    "END_AGPA_PERCENT",
    "END_AGPA_POINTS",
    "SEMESTER_REG_COURSES",
    "SEMESTER_REG_CREDITS",
    "SEMESTER_PASS_COURSES",
    "SEMESTER_PASS_CREDITS",
    "SEMESTER_FAIL_COURSES",
    "SEMESTER_FAIL_CREDITS",
    "TOTAL_PASS_COURSES",
    "TOTAL_PASS_CREDITS",
    "TOTAL_FAIL_COURSES",
    "TOTAL_FAIL_CREDITS",
    "REG_TOTAL_SEMESTERS",
    "FINISH_STATUS",
    "VERSION_TITLE_SL",
    "START_LEVEL_ID",
    "START_LEVEL_NAME_PL",
]

query = f"""
SELECT
    STUDENT_STATUS_ID,
    STUDENT_ID,
    PART_ID,
    DEGREE_ID,
    START_PART_ID,
    FINISH_PART_ID,
    GRADE_VERSION_ID,
    PERMANENT_STATUS_ID,
    PERMANENT_STATUS_SL,
    STUDY_MODE,
    PREV_GPA_POINTS,
    PREV_GPA_PERCENT,
    GPA_PERCENT,
    GPA_POINTS,
    START_AGPA_PERCENT,
    START_AGPA_POINTS,
    START_TOTAL_IN_COURSES,
    START_TOTAL_IN_CREDITS,
    END_TOTAL_IN_COURSES,
    END_TOTAL_IN_CREDITS,
    END_AGPA_PERCENT,
    END_AGPA_POINTS,
    SEMESTER_REG_COURSES,
    SEMESTER_REG_CREDITS,
    SEMESTER_PASS_COURSES,  
    SEMESTER_PASS_CREDITS,
    SEMESTER_FAIL_COURSES,
    SEMESTER_FAIL_CREDITS,
    SEMESTER_IN_COURSES,
    SEMESTER_IN_CREDITS,
    TOTAL_SEMESTERS,
    TOTAL_REG_COURSES,
    TOTAL_REG_CREDITS,
    TOTAL_PASS_COURSES,
    TOTAL_PASS_CREDITS,
    TOTAL_FAIL_COURSES,
    TOTAL_FAIL_CREDITS,
    REG_TOTAL_SEMESTERS,
    FINISH_STATUS,
    VERSION_TITLE_SL,
    DEGREE_NAME_SL,
    DEGREE_CREDITS_COUNT,
    START_LEVEL_ID,
    START_LEVEL_NAME_PL
FROM {VIEW_NAME}
WHERE STUDY_MODE = 'C'
AND ACTIVE = 'A'
"""

df_v_add_student_degree_status = pd.read_sql(query, connection)
df_v_add_student_degree_status.columns = df_v_add_student_degree_status.columns.str.lower()

save_parquet(df_v_add_student_degree_status, RAW_DIR / "v_add_student_degree_status.parquet")

print("Rows loaded:", len(df_v_add_student_degree_status))
print("Columns:", df_v_add_student_degree_status.columns.tolist())
print("Saved:", RAW_DIR / "v_add_student_degree_status.parquet")

display(df_v_add_student_degree_status.head())

C:\Users\ASUS\AppData\Local\Temp\ipykernel_20092\106809846.py:82: UserWarning: pandas only supports SQLAlchemy connectable (engine/connection) or database string URI or sqlite3 DBAPI2 connection. Other DBAPI2 objects are not tested. Please consider using SQLAlchemy.
  df_v_add_student_degree_status = pd.read_sql(query, connection)


Rows loaded: 189158
Columns: ['student_status_id', 'student_id', 'part_id', 'degree_id', 'start_part_id', 'finish_part_id', 'grade_version_id', 'permanent_status_id', 'permanent_status_sl', 'study_mode', 'prev_gpa_points', 'prev_gpa_percent', 'gpa_percent', 'gpa_points', 'start_agpa_percent', 'start_agpa_points', 'start_total_in_courses', 'start_total_in_credits', 'end_total_in_courses', 'end_total_in_credits', 'end_agpa_percent', 'end_agpa_points', 'semester_reg_courses', 'semester_reg_credits', 'semester_pass_courses', 'semester_pass_credits', 'semester_fail_courses', 'semester_fail_credits', 'semester_in_courses', 'semester_in_credits', 'total_semesters', 'total_reg_courses', 'total_reg_credits', 'total_pass_courses', 'total_pass_credits', 'total_fail_courses', 'total_fail_credits', 'reg_total_semesters', 'finish_status', 'version_title_sl', 'degree_name_sl', 'degree_credits_count', 'start_level_id', 'start_level_name_pl']
Saved: D:\AI\Real projects\Academic_Advisor\data\raw\v_add_s

,student_status_id,student_id,part_id,degree_id,start_part_id,finish_part_id,grade_version_id,permanent_status_id,permanent_status_sl,study_mode,prev_gpa_points,prev_gpa_percent,gpa_percent,gpa_points,start_agpa_percent,start_agpa_points,start_total_in_courses,start_total_in_credits,end_total_in_courses,end_total_in_credits,end_agpa_percent,end_agpa_points,semester_reg_courses,semester_reg_credits,semester_pass_courses,semester_pass_credits,semester_fail_courses,semester_fail_credits,semester_in_courses,semester_in_credits,total_semesters,total_reg_courses,total_reg_credits,total_pass_courses,total_pass_credits,total_fail_courses,total_fail_credits,reg_total_semesters,finish_status,version_title_sl,degree_name_sl,degree_credits_count,start_level_id,start_level_name_pl
0,297550.111,48.111,20203.0,22.111,20203.0,20212.0,3.111,2.0,مقبول,C,0.00,0.0,0.0,0.00,39.4,0.97,26.0,66.0,26.0,66.0,38.6,0.93,2.0,4.0,0.0,0.0,2.0,4.0,0.0,0.0,16.0,73.0,180.0,26.0,66.0,47.0,114.0,12.0,CLOSE_FILE,القرار رقم 230,إدارة (اختصاص عام),72.0,692.111,Third year
1,314343.111,48.111,20211.0,22.111,20203.0,20212.0,3.111,2.0,مقبول,C,0.00,0.0,0.0,0.00,38.6,0.93,26.0,66.0,26.0,66.0,38.6,0.93,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,17.0,75.0,184.0,26.0,66.0,49.0,118.0,12.0,CLOSE_FILE,القرار رقم 230,إدارة (اختصاص عام),72.0,692.111,Third year
2,320948.111,48.111,20212.0,22.111,20203.0,20212.0,3.111,17.0,ترقين قيد,C,0.00,0.0,0.0,0.00,38.6,0.93,26.0,66.0,26.0,66.0,38.6,0.93,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,18.0,75.0,184.0,26.0,66.0,49.0,118.0,12.0,CLOSE_FILE,القرار رقم 230,إدارة (اختصاص عام),72.0,692.111,Third year
3,114381.111,128.111,20111.0,19.111,20111.0,20151.0,2.111,2.0,مقبول,C,NaN,NaN,51.4,1.57,20.0,0.00,0.0,0.0,5.0,13.0,51.4,1.57,7.0,17.0,5.0,13.0,2.0,4.0,5.0,13.0,1.0,0.0,0.0,0.0,0.0,0.0,0.0,1.0,CLOSE_FILE,القرار الوزاري الموحد,إدارة الأعمال,148.0,236.111,First Year
4,114382.111,128.111,20112.0,19.111,20111.0,20151.0,2.111,2.0,مقبول,C,1.57,51.4,31.6,0.58,51.4,1.57,5.0,13.0,7.0,18.0,41.2,1.06,8.0,18.0,2.0,5.0,6.0,13.0,2.0,5.0,2.0,7.0,17.0,5.0,13.0,2.0,4.0,2.0,CLOSE_FILE,القرار الوزاري الموحد,إدارة الأعمال,148.0,236.111,First Year


In [10]:
VIEW_NAME = "RAS_USER.V_ACD_DEGREE_COURSE"

needed_columns = [
    "DEGREE_COURSE_ID",
    "COURSE_ID",
    "COURSE_NAME_SL",
    "DEGREE_ID",
    "DEGREE_NAME_SL",
    "YEAR_ORDER",
    "SEMESTER_ORDER",
    "COURSE_CREDITS",
    "ACTIVE",
    "CREDITS_COUNT",
    "FACULTY_ID",
    "REQUIRED_CREDITS",
    "REQUIREMENT_TYPE_ID",
    "REQUIREMENT_TYPE_SL",
    "REQ_DEGREE_ID",
]

query = f"""
SELECT
    DEGREE_COURSE_ID,
    COURSE_ID,
    DEGREE_ID,
    COURSE_TYPE_ID,
    REQUIREMENT_TYPE_ID,
    REQUIREMENT_TYPE_SL,
    COURSE_NAME_SL,
    COURSE_OFFICIAL_SL,
    DEGREE_NAME_SL,
    YEAR_ORDER,
    SEMESTER_ORDER,
    COURSE_CREDITS,
    ACTIVE,
    CREDITS_COUNT
FROM {VIEW_NAME}
WHERE ACTIVE = 'A'
"""

df_v_acd_degree_course = pd.read_sql(query, connection)
df_v_acd_degree_course.columns = df_v_acd_degree_course.columns.str.lower()

save_parquet(df_v_acd_degree_course, RAW_DIR / "v_acd_degree_course.parquet")

print("Rows loaded:", len(df_v_acd_degree_course))
print("Columns:", df_v_acd_degree_course.columns.tolist())
print("Saved:", RAW_DIR / "v_acd_degree_course.parquet")

display(df_v_acd_degree_course.head())

C:\Users\ASUS\AppData\Local\Temp\ipykernel_20092\4230206980.py:41: UserWarning: pandas only supports SQLAlchemy connectable (engine/connection) or database string URI or sqlite3 DBAPI2 connection. Other DBAPI2 objects are not tested. Please consider using SQLAlchemy.
  df_v_acd_degree_course = pd.read_sql(query, connection)


Rows loaded: 4006
Columns: ['degree_course_id', 'course_id', 'degree_id', 'course_type_id', 'requirement_type_id', 'requirement_type_sl', 'course_name_sl', 'course_official_sl', 'degree_name_sl', 'year_order', 'semester_order', 'course_credits', 'active', 'credits_count']
Saved: D:\AI\Real projects\Academic_Advisor\data\raw\v_acd_degree_course.parquet


,degree_course_id,course_id,degree_id,course_type_id,requirement_type_id,requirement_type_sl,course_name_sl,course_official_sl,degree_name_sl,year_order,semester_order,course_credits,active,credits_count
0,1.111,917.111,17.111,3.0,5.0,متطلبات الشهادة الإجبارية,الكيمياء الحيوية,الكيمياء الحيوية,الصيدلة,3.0,1.0,4.0,A,193.0
1,2.111,923.111,17.111,3.0,5.0,متطلبات الشهادة الإجبارية,الكيمياء الحيوية التطبيقية,الكيمياء الحيوية التطبيقية,الصيدلة,3.0,2.0,3.0,A,193.0
2,3.111,908.111,17.111,1.0,5.0,متطلبات الشهادة الإجبارية,الإحصاء الحيوي,الإحصاء الحيوي,الصيدلة,2.0,1.0,2.0,A,193.0
3,4.111,891.111,17.111,3.0,5.0,متطلبات الشهادة الإجبارية,البيولوجيا النباتية,البيولوجيا النباتية,الصيدلة,1.0,1.0,4.0,A,193.0
4,5.111,907.111,17.111,1.0,5.0,متطلبات الشهادة الإجبارية,تجارة الأدوية,تجارة الأدوية,الصيدلة,2.0,1.0,2.0,A,193.0


In [11]:
VIEW_NAME = "RAS_USER.V_CRG_STUDENT_PASSED_CREDIT"

query = f"""
SELECT
    STUDENT_ID,
    REQUIREMENT_TYPE_PL,
    REQUIREMENT_TYPE_SL,
    CREDITS_COUNT,
    PASSED_CREDIT
FROM {VIEW_NAME}
"""

df_v_crg_student_passed_credit = pd.read_sql(query, connection)
df_v_crg_student_passed_credit.columns = df_v_crg_student_passed_credit.columns.str.lower()

save_parquet(df_v_crg_student_passed_credit, RAW_DIR / "v_crg_student_passed_credit.parquet")

print("Rows loaded:", len(df_v_crg_student_passed_credit))
print("Columns:", df_v_crg_student_passed_credit.columns.tolist())
print("Saved:", RAW_DIR / "v_crg_student_passed_credit.parquet")

display(df_v_crg_student_passed_credit.head())

C:\Users\ASUS\AppData\Local\Temp\ipykernel_20092\207756106.py:13: UserWarning: pandas only supports SQLAlchemy connectable (engine/connection) or database string URI or sqlite3 DBAPI2 connection. Other DBAPI2 objects are not tested. Please consider using SQLAlchemy.
  df_v_crg_student_passed_credit = pd.read_sql(query, connection)


Rows loaded: 85358
Columns: ['student_id', 'requirement_type_pl', 'requirement_type_sl', 'credits_count', 'passed_credit']
Saved: D:\AI\Real projects\Academic_Advisor\data\raw\v_crg_student_passed_credit.parquet


,student_id,requirement_type_pl,requirement_type_sl,credits_count,passed_credit
0,16.111,University Mandatory Requirements,متطلبات الجامعة الإجبارية,20.0,20.0
1,35.111,Mandatory Degree Requirements,متطلبات الشهادة الإجبارية,128.0,94.0
2,46.111,Mandatory Degree Requirements,متطلبات الشهادة الإجبارية,128.0,90.0
3,69.111,Mandatory Degree Requirements,متطلبات الشهادة الإجبارية,128.0,128.0
4,80.111,University Mandatory Requirements,متطلبات الجامعة الإجبارية,20.0,20.0


In [12]:
VIEW_NAME = "RAS_USER.V_SCH_COURSE_OFFER"

query = f"""
SELECT
    LEVEL_CATEGORY_ID,
    PART_ID,
    DEPARTMENT_ID,
    FACULTY_ID,
    COURSE_ID,
    COURSE_TYPE_ID,
    COURSE_NAME_SL,
    FACULTY_NAME_SL,
    DEPARTMENT_NAME_SL,
    COURSE_CREDITS,
    COURSE_STATUS,
    ALLOW_REGISTER
FROM {VIEW_NAME}
WHERE ACTIVE = 'A'
"""

df_v_sch_course_offers = pd.read_sql(query, connection)
df_v_sch_course_offers.columns = df_v_sch_course_offers.columns.str.lower()

save_parquet(df_v_sch_course_offers, RAW_DIR / "v_sch_course_offers.parquet")

print("Rows loaded:", len(df_v_sch_course_offers))
print("Columns:", df_v_sch_course_offers.columns.tolist())
print("Saved:", RAW_DIR / "v_sch_course_offers.parquet")

display(df_v_sch_course_offers.head())

C:\Users\ASUS\AppData\Local\Temp\ipykernel_20092\4196843430.py:21: UserWarning: pandas only supports SQLAlchemy connectable (engine/connection) or database string URI or sqlite3 DBAPI2 connection. Other DBAPI2 objects are not tested. Please consider using SQLAlchemy.
  df_v_sch_course_offers = pd.read_sql(query, connection)


Rows loaded: 164737
Columns: ['level_category_id', 'part_id', 'department_id', 'faculty_id', 'course_id', 'course_type_id', 'course_name_sl', 'faculty_name_sl', 'department_name_sl', 'course_credits', 'course_status', 'allow_register']
Saved: D:\AI\Real projects\Academic_Advisor\data\raw\v_sch_course_offers.parquet


,level_category_id,part_id,department_id,faculty_id,course_id,course_type_id,course_name_sl,faculty_name_sl,department_name_sl,course_credits,course_status,allow_register
0,1.0,20050.0,40.111,7.111,1.111,1.0,مبادئ المحاسبة 1,كلية إدارة الأعمال,قسم المحاسبة و التدقيق,2.0,NaN,NaN
1,1.0,20051.0,40.111,7.111,1.111,1.0,مبادئ المحاسبة 1,كلية إدارة الأعمال,قسم المحاسبة و التدقيق,2.0,NaN,NaN
2,1.0,20052.0,40.111,7.111,1.111,1.0,مبادئ المحاسبة 1,كلية إدارة الأعمال,قسم المحاسبة و التدقيق,2.0,NaN,NaN
3,1.0,20053.0,40.111,7.111,1.111,1.0,مبادئ المحاسبة 1,كلية إدارة الأعمال,قسم المحاسبة و التدقيق,2.0,NaN,NaN
4,1.0,20060.0,40.111,7.111,1.111,1.0,مبادئ المحاسبة 1,كلية إدارة الأعمال,قسم المحاسبة و التدقيق,2.0,NaN,NaN


In [13]:
VIEW_NAME = "RAS_USER.V_CRG_STD_COR_TEMP_REQUEST"

needed_columns = [
"ACTIVE",
"REQUIREMENT_TYPE_ID",
"IS_REQUESTABLE",
"PASSED_PREREQUISITES",
"LAST_REGISTER_SEMESTER"
"STATUS_REASON_SL",
"STATUS_REASON_CODE",
"FINAL_MARK",
"SEMESTER_ORDER",
"YEAR_ORDER",
"COURSE_CREITS",
"COURSE_NAME_SL", 
"COURSE_ID",
"CREDITS_COUNT",
"CREDITS_TILL_GRAD",
"STUDENT_ID",
"STD_COR_TEMP_REQUEST_ID"
]

query = f"""
SELECT
    STD_COR_TEMP_REQUEST_ID,
    STUDENT_ID,
    PART_ID,
    COURSE_ID,
    COURSE_TYPE_ID,
    REQUIREMENT_TYPE_ID,

    ALLOW_REGISTER,
    IS_REQUESTABLE,
    GPA_PERCENT,
    GPA_POINTS,
    END_AGPA_POINTS,
    CREDITS_TILL_GRAD,
    REQUIREMENT_PASSED_CREDITS,
    CREDITS_COUNT,
    COURSE_CREDITS,
    YEAR_ORDER,
    SEMESTER_ORDER,
    REGISTER_STATUS,
    FINISH_STATUS
FROM RAS_USER.V_CRG_STD_COR_TEMP_REQUEST
ORDER BY
    STUDENT_ID,
    PART_ID,
    COURSE_ID,
    REQUIREMENT_TYPE_ID
"""

df_vcrg_std_cor_temp_request = pd.read_sql(query, connection)
df_vcrg_std_cor_temp_request.columns = df_vcrg_std_cor_temp_request.columns.str.lower()

save_parquet(df_vcrg_std_cor_temp_request, RAW_DIR / "v_crg_std_cor_temp_request.parquet")

print("Rows loaded:", len(df_vcrg_std_cor_temp_request))
print("Columns:", df_vcrg_std_cor_temp_request.columns.tolist())
print("Saved:", RAW_DIR / "v_crg_std_cor_temp_request.parquet")

display(df_vcrg_std_cor_temp_request.head())

C:\Users\ASUS\AppData\Local\Temp\ipykernel_20092\3222691328.py:53: UserWarning: pandas only supports SQLAlchemy connectable (engine/connection) or database string URI or sqlite3 DBAPI2 connection. Other DBAPI2 objects are not tested. Please consider using SQLAlchemy.
  df_vcrg_std_cor_temp_request = pd.read_sql(query, connection)


Rows loaded: 556700
Columns: ['std_cor_temp_request_id', 'student_id', 'part_id', 'course_id', 'course_type_id', 'requirement_type_id', 'allow_register', 'is_requestable', 'gpa_percent', 'gpa_points', 'end_agpa_points', 'credits_till_grad', 'requirement_passed_credits', 'credits_count', 'course_credits', 'year_order', 'semester_order', 'register_status', 'finish_status']
Saved: D:\AI\Real projects\Academic_Advisor\data\raw\v_crg_std_cor_temp_request.parquet


,std_cor_temp_request_id,student_id,part_id,course_id,course_type_id,requirement_type_id,allow_register,is_requestable,gpa_percent,gpa_points,end_agpa_points,credits_till_grad,requirement_passed_credits,credits_count,course_credits,year_order,semester_order,register_status,finish_status
0,4611.0,40.111,20081.0,149.111,1.0,1.0,N,N,0.0,0.0,2.24,5.0,20.0,20.0,2.0,1.0,1.0,R,P
1,4641.0,40.111,20081.0,150.111,1.0,1.0,N,N,0.0,0.0,2.24,5.0,20.0,20.0,3.0,1.0,1.0,R,P
2,4624.0,40.111,20081.0,151.111,1.0,1.0,N,N,0.0,0.0,2.24,5.0,20.0,20.0,3.0,1.0,1.0,R,P
3,4645.0,40.111,20081.0,152.111,1.0,5.0,N,N,0.0,0.0,2.24,5.0,123.0,128.0,2.0,1.0,1.0,R,P
4,4612.0,40.111,20081.0,153.111,1.0,5.0,N,N,0.0,0.0,2.24,5.0,123.0,128.0,2.0,1.0,1.0,R,P


In [14]:
VIEW_NAME= "RAS_USER.V_COR_COURSE_PREREQUISITE "
query = f"""
SELECT*
FROM {VIEW_NAME}
"""
df_v_cor_course_prerequisite = pd.read_sql(query, connection)
df_v_cor_course_prerequisite.columns = df_v_cor_course_prerequisite.columns.str.lower()

save_parquet(df_v_cor_course_prerequisite, RAW_DIR / "v_cor_course_prerequisite.parquet")

print("Rows loaded:", len(df_v_cor_course_prerequisite))
print("Columns:", df_v_cor_course_prerequisite.columns.tolist())
print("Saved:", RAW_DIR / "v_cor_course_prerequisite.parquet")
display(df_v_cor_course_prerequisite.head())

C:\Users\ASUS\AppData\Local\Temp\ipykernel_20092\152342875.py:6: UserWarning: pandas only supports SQLAlchemy connectable (engine/connection) or database string URI or sqlite3 DBAPI2 connection. Other DBAPI2 objects are not tested. Please consider using SQLAlchemy.
  df_v_cor_course_prerequisite = pd.read_sql(query, connection)


Rows loaded: 1306
Columns: ['course_prerequisite_id', 'course_id', 'prerequisite_id', 'group_num', 'course_name_pl', 'course_name_sl', 'course_code', 'degree_id', 'active']
Saved: D:\AI\Real projects\Academic_Advisor\data\raw\v_cor_course_prerequisite.parquet


,course_prerequisite_id,course_id,prerequisite_id,group_num,course_name_pl,course_name_sl,course_code,degree_id,active
0,17.111,2.111,1.111,1,Accounting Principles I,مبادئ المحاسبة 1,BAAA-1-01,NaN,A
1,54.111,44.111,1.111,1,Accounting Principles I,مبادئ المحاسبة 1,BAAA-1-01,NaN,A
2,18.111,3.111,2.111,1,Accounting Principles II,مبادئ المحاسبة 2,BAAA-2-02,NaN,A
3,19.111,4.111,2.111,1,Accounting Principles II,مبادئ المحاسبة 2,BAAA-2-02,NaN,A
4,20.111,5.111,2.111,1,Accounting Principles II,مبادئ المحاسبة 2,BAAA-2-02,NaN,A


In [15]:
cursor.close()